# Poisson Process Distribution Family
### Poisson → Exponential → Erlang → Gamma

All four distributions come from the same idea: a **Poisson process** — events happening randomly at a constant rate λ.

| Distribution | Question it answers |
|---|---|
| Poisson | How many events in a fixed interval? |
| Exponential | How long until the 1st event? |
| Erlang | How long until the k-th event? |
| Gamma | Erlang but k can be any positive number |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.special import factorial

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f9f9f9'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['font.size'] = 11
print('Libraries loaded.')

---
## 1. Poisson Distribution

**Question:** How many emails arrive in 1 hour?

**Formula:**
$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

**Parameters:**
- λ = average number of events per interval
- k = exact count we are asking about (0, 1, 2, 3 …)

**Example:** You receive on average **5 emails per hour**. What is the probability of getting exactly 3 emails in the next hour?

In [ ]:
lam = 5  # average emails per hour
k_vals = np.arange(0, 15)
poisson_probs = stats.poisson.pmf(k_vals, lam)

# Exact answer for k=3
k = 3
p_exact = stats.poisson.pmf(k, lam)
print(f'λ = {lam} emails/hour')
print(f'P(X = {k}) = {p_exact:.4f}  →  {p_exact*100:.2f}% chance of exactly {k} emails')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# PMF
colors = ['#E74C3C' if k == 3 else '#3498DB' for k in k_vals]
axes[0].bar(k_vals, poisson_probs, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title(f'Poisson PMF  (λ = {lam})', fontweight='bold')
axes[0].set_xlabel('Number of emails (k)')
axes[0].set_ylabel('Probability P(X = k)')
axes[0].annotate(f'P(X=3) = {p_exact:.3f}', xy=(3, p_exact), xytext=(6, p_exact+0.02),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

# Effect of different lambdas
for l, color in zip([2, 5, 10], ['#2ECC71', '#3498DB', '#9B59B6']):
    axes[1].plot(k_vals, stats.poisson.pmf(k_vals, l), 'o-', label=f'λ = {l}', color=color, linewidth=1.5, markersize=5)
axes[1].set_title('Poisson PMF for different λ values', fontweight='bold')
axes[1].set_xlabel('Number of events (k)')
axes[1].set_ylabel('Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. Exponential Distribution

**Question:** How long until the next email arrives?

**Formula:**
$$P(X \leq t) = 1 - e^{-\lambda t} \quad \text{(within t)}$$
$$P(X > t) = e^{-\lambda t} \quad \text{(more than t)}$$

**Where it comes from:** Plug k = 0 into Poisson:
$$P(\text{0 events in time } t) = \frac{(\lambda t)^0 e^{-\lambda t}}{0!} = e^{-\lambda t}$$

**Key property:** Memoryless — past waiting time does not affect future wait.

**Example:** Emails arrive at **λ = 5 per hour**. What is the probability the next email arrives within 10 minutes (t = 1/6 hour)?

In [ ]:
lam = 5  # emails per hour
t_vals = np.linspace(0, 1.5, 300)  # in hours

# P(X <= 10 min)
t_ask = 10/60  # 10 minutes in hours
p_within = 1 - np.exp(-lam * t_ask)
p_more   = np.exp(-lam * t_ask)
print(f'λ = {lam} emails/hour,  mean wait = {1/lam*60:.1f} minutes')
print(f'P(next email within 10 min) = {p_within:.4f}  →  {p_within*100:.1f}%')
print(f'P(still waiting after 10 min) = {p_more:.4f}  →  {p_more*100:.1f}%')

pdf_vals = lam * np.exp(-lam * t_vals)
cdf_vals = 1 - np.exp(-lam * t_vals)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# PDF with shaded region
axes[0].plot(t_vals * 60, pdf_vals / 60, color='#E67E22', linewidth=2)
mask = t_vals <= t_ask
axes[0].fill_between(t_vals[mask]*60, pdf_vals[mask]/60, alpha=0.35, color='#E67E22', label=f'P(X ≤ 10 min) = {p_within:.3f}')
axes[0].fill_between(t_vals[~mask]*60, pdf_vals[~mask]/60, alpha=0.15, color='#3498DB', label=f'P(X > 10 min) = {p_more:.3f}')
axes[0].axvline(10, color='red', linestyle='--', linewidth=1.5, label='t = 10 min')
axes[0].set_title(f'Exponential PDF  (λ = {lam}/hr)', fontweight='bold')
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 90)

# CDF
axes[1].plot(t_vals * 60, cdf_vals, color='#E67E22', linewidth=2)
axes[1].axvline(10, color='red', linestyle='--', linewidth=1.5)
axes[1].axhline(p_within, color='red', linestyle='--', linewidth=1.5)
axes[1].plot(10, p_within, 'ro', markersize=8, zorder=5)
axes[1].annotate(f'{p_within:.3f}', xy=(10, p_within), xytext=(25, p_within-0.08), fontsize=10, color='red')
axes[1].set_title('Exponential CDF  P(X ≤ t)', fontweight='bold')
axes[1].set_xlabel('Time (minutes)')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_xlim(0, 90)

plt.tight_layout()
plt.show()

---
## 3. Erlang Distribution

**Question:** How long until the k-th email arrives?

**Formula:**
$$P(X \leq t) = 1 - \sum_{n=0}^{k-1} \frac{(\lambda t)^n e^{-\lambda t}}{n!}$$

**Intuition:** Erlang(k, λ) = sum of k independent Exponential(λ) waiting times.

**Key constraint:** k must be a **positive integer** (1, 2, 3 …)

**Note:** Erlang(k=1, λ) = Exponential(λ) — Exponential is just a special case!

**Example:** Emails arrive at **λ = 5 per hour**. How long to wait for the **3rd email**?

In [ ]:
lam = 5  # emails per hour
t_vals = np.linspace(0, 2.5, 400)

# Erlang is Gamma with integer shape
k_vals_erlang = [1, 2, 3, 5]

# Example: k=3
k = 3
mean_wait = k / lam
print(f'λ = {lam} emails/hour,  k = {k} (waiting for 3rd email)')
print(f'Mean wait for {k}rd email = {k}/{lam} hr = {mean_wait*60:.1f} minutes')
t_ask = 30/60  # 30 minutes
p_within_30 = stats.gamma.cdf(t_ask, a=k, scale=1/lam)
print(f'P(3rd email arrives within 30 min) = {p_within_30:.4f}  →  {p_within_30*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors_k = ['#E74C3C', '#E67E22', '#9B59B6', '#27AE60']

for k_e, c in zip(k_vals_erlang, colors_k):
    pdf = stats.gamma.pdf(t_vals, a=k_e, scale=1/lam)
    axes[0].plot(t_vals * 60, pdf / 60, color=c, linewidth=2, label=f'k = {k_e}  (mean = {k_e/lam*60:.0f} min)')

axes[0].set_title(f'Erlang PDF  (λ = {lam}/hr, varying k)', fontweight='bold')
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 150)

# Shade k=3 example
k3_pdf = stats.gamma.pdf(t_vals, a=3, scale=1/lam)
mask = t_vals <= t_ask
axes[1].plot(t_vals * 60, k3_pdf / 60, color='#9B59B6', linewidth=2)
axes[1].fill_between(t_vals[mask]*60, k3_pdf[mask]/60, alpha=0.4, color='#9B59B6',
                     label=f'P(X ≤ 30 min) = {p_within_30:.3f}')
axes[1].fill_between(t_vals[~mask]*60, k3_pdf[~mask]/60, alpha=0.15, color='#9B59B6')
axes[1].axvline(30, color='red', linestyle='--', linewidth=1.5, label='t = 30 min')
axes[1].axvline(k/lam*60, color='gray', linestyle=':', linewidth=1.5, label=f'mean = {k/lam*60:.0f} min')
axes[1].set_title('Erlang PDF  (k=3, λ=5/hr)', fontweight='bold')
axes[1].set_xlabel('Time (minutes)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 150)

plt.tight_layout()
plt.show()

---
## 4. Gamma Distribution

**Question:** Same as Erlang but α (shape) can be any positive real number.

**Formula (PDF):**
$$f(x) = \frac{\lambda^\alpha x^{\alpha-1} e^{-\lambda x}}{\Gamma(\alpha)}$$

**Where** $\Gamma(\alpha)$ is the Gamma function — a scaling factor ensuring total probability = 1.

**Special cases:**
- α = 1 → **Exponential** distribution
- α = integer → **Erlang** distribution
- α = any positive real → **Gamma** (most general)

**Used for:** modelling skewed waiting times, insurance claims, rainfall, income distributions — anywhere Erlang is too restrictive.

**Example:** Fitting real-world email response time data where α = 2.5 (not a whole number).

In [ ]:
lam = 5
t_vals = np.linspace(0.001, 2.5, 400)

# Integer shapes = Erlang, non-integer = true Gamma
shapes = [0.5, 1, 2, 2.5, 5]
colors_g = ['#1ABC9C', '#E74C3C', '#E67E22', '#3498DB', '#9B59B6']
labels  = ['α = 0.5', 'α = 1 (Exponential)', 'α = 2 (Erlang k=2)', 'α = 2.5 (true Gamma)', 'α = 5 (Erlang k=5)']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for a, c, lb in zip(shapes, colors_g, labels):
    pdf = stats.gamma.pdf(t_vals, a=a, scale=1/lam)
    axes[0].plot(t_vals * 60, pdf / 60, color=c, linewidth=2, label=lb)

axes[0].set_title(f'Gamma PDF  (λ = {lam}/hr, varying α)', fontweight='bold')
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8.5)
axes[0].set_xlim(0, 150)
axes[0].set_ylim(0)

# CDF comparison
for a, c, lb in zip(shapes, colors_g, labels):
    cdf = stats.gamma.cdf(t_vals, a=a, scale=1/lam)
    axes[1].plot(t_vals * 60, cdf, color=c, linewidth=2, label=lb)

axes[1].set_title('Gamma CDF  P(X ≤ t)', fontweight='bold')
axes[1].set_xlabel('Time (minutes)')
axes[1].set_ylabel('Cumulative Probability')
axes[1].legend(fontsize=8.5)
axes[1].set_xlim(0, 150)

plt.tight_layout()
plt.show()

---
## 5. All Four Together — The Full Family

Same scenario throughout: **emails arriving at λ = 5 per hour**

In [ ]:
lam = 5
t_vals = np.linspace(0.001, 2, 400)
k_vals = np.arange(0, 14)

fig = plt.figure(figsize=(14, 10))
fig.suptitle('The Poisson Process Family  (λ = 5 events/hour)', fontsize=14, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

# ── Poisson ──
ax1 = fig.add_subplot(gs[0, 0])
pmf = stats.poisson.pmf(k_vals, lam)
bar_colors = ['#E74C3C' if k == 5 else '#3498DB' for k in k_vals]
ax1.bar(k_vals, pmf, color=bar_colors, edgecolor='white')
ax1.set_title('Poisson\nHow many events in 1 hour?', fontweight='bold', fontsize=10)
ax1.set_xlabel('Count (k)')
ax1.set_ylabel('P(X = k)')
ax1.text(7, 0.16, 'red bar = k=5\n(most likely)', color='#E74C3C', fontsize=8)

# ── Exponential ──
ax2 = fig.add_subplot(gs[0, 1])
pdf_exp = stats.gamma.pdf(t_vals, a=1, scale=1/lam)
t10 = 10/60
mask = t_vals <= t10
ax2.plot(t_vals*60, pdf_exp/60, color='#E67E22', linewidth=2)
ax2.fill_between(t_vals[mask]*60, pdf_exp[mask]/60, alpha=0.4, color='#E67E22')
ax2.axvline(t10*60, color='red', linestyle='--', linewidth=1.5)
p10 = 1 - np.exp(-lam*t10)
ax2.set_title(f'Exponential  (α=1)\nWait for 1st event  [P(≤10min)={p10:.2f}]', fontweight='bold', fontsize=10)
ax2.set_xlabel('Time (minutes)')
ax2.set_ylabel('Density')
ax2.set_xlim(0, 90)

# ── Erlang ──
ax3 = fig.add_subplot(gs[1, 0])
for k_e, c, ls in zip([1,2,3,5], ['#E74C3C','#E67E22','#9B59B6','#27AE60'], ['-','--','-.',':']):
    pdf_e = stats.gamma.pdf(t_vals, a=k_e, scale=1/lam)
    ax3.plot(t_vals*60, pdf_e/60, color=c, linewidth=2, linestyle=ls, label=f'k={k_e}')
ax3.set_title('Erlang  (integer α)\nWait for k-th event', fontweight='bold', fontsize=10)
ax3.set_xlabel('Time (minutes)')
ax3.set_ylabel('Density')
ax3.legend(fontsize=9)
ax3.set_xlim(0, 150)

# ── Gamma ──
ax4 = fig.add_subplot(gs[1, 1])
for a, c, ls in zip([1, 2, 2.5, 5], ['#E74C3C','#E67E22','#3498DB','#9B59B6'], ['-','--','-.',':']):
    pdf_g = stats.gamma.pdf(t_vals, a=a, scale=1/lam)
    lbl = f'α={a}' + (' ← non-integer' if a == 2.5 else '')
    ax4.plot(t_vals*60, pdf_g/60, color=c, linewidth=2, linestyle=ls, label=lbl)
ax4.set_title('Gamma  (any α > 0)\nGeneralized wait time', fontweight='bold', fontsize=10)
ax4.set_xlabel('Time (minutes)')
ax4.set_ylabel('Density')
ax4.legend(fontsize=9)
ax4.set_xlim(0, 150)

plt.tight_layout()
plt.show()

print('\nRelationship summary:')
print('  Exponential = Erlang(k=1) = Gamma(α=1)')
print('  Erlang(k)   = Gamma(α=k) where k is a positive integer')
print('  Gamma(α)    = most general form, α can be any positive real number')

---
## 6. Quick Reference — Formulas & Properties

| | Poisson | Exponential | Erlang | Gamma |
|---|---|---|---|---|
| **Type** | Discrete | Continuous | Continuous | Continuous |
| **Parameter** | λ | λ | k (int), λ | α (real), λ |
| **Mean** | λ | 1/λ | k/λ | α/λ |
| **Question** | How many? | Wait for 1st | Wait for k-th | Generalized wait |
| **Special case of** | — | Erlang k=1 | Gamma α=integer | Most general |

### Memoryless Property (Exponential only)

$$P(X > s + t \mid X > s) = P(X > t)$$

No matter how long you have already waited, the remaining wait has the same distribution. This is unique to the Exponential distribution among all continuous distributions.